# Careem Partner Analytics — Linear Regression Forecast Model
> **SYNTHETIC DATA DISCLAIMER**: All numbers generated and analyzed in this notebook are 100% synthetic and simulated for educational and portfolio presentation purposes as part of the SafeX Solutions Internship (Week 4, Group 56).

## Notebook Objectives:
1. Load cleaned dataset (`data/careem_partner_monthly_clean.csv`)
2. Feature engineering: `time_idx`, cyclical month signals (`sin_month`, `cos_month`), `active_promo_days`, `rainy_days`, `is_ramadan_month`
3. Chronological train/test split (18 train months / 6 test months)
4. Train `scikit-learn` LinearRegression models for `orders` and `revenue_pkr`
5. Evaluate out-of-sample MAE & MAPE metrics on the 6 test months
6. Generate 3-month forward out-of-sample forecast and save to `model/outputs/regression_forecast.csv`

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

clean_data_path = "../data/careem_partner_monthly_clean.csv"
df = pd.read_csv(clean_data_path)
df['date'] = pd.to_datetime(df['month'])
df['time_idx'] = np.arange(len(df))
df['month_num'] = df['date'].dt.month
df['sin_month'] = np.sin(2 * np.pi * df['month_num'] / 12)
df['cos_month'] = np.cos(2 * np.pi * df['month_num'] / 12)

feature_cols = ['time_idx', 'sin_month', 'cos_month', 'active_promo_days', 'rainy_days', 'is_ramadan_month']
df.head()

### Temporal Train/Test Split (18 Train / 6 Test)
*Note: A temporal split is used instead of random K-fold CV to preserve time-series sequential order and prevent data leakage.*

In [ ]:
train_df = df.iloc[:18]
test_df = df.iloc[18:]

model_orders = LinearRegression()
model_orders.fit(train_df[feature_cols], train_df['orders'])

test_preds = model_orders.predict(test_df[feature_cols])
actuals = test_df['orders'].values

mae = np.mean(np.abs(actuals - test_preds))
mape = np.mean(np.abs((actuals - test_preds) / actuals)) * 100

print(f"Regression Test Orders -> MAE: {mae:.2f}, MAPE: {mape:.2f}%")